# LLM Hawkish/Dovish Classification — Excel edition
8 models × 5 prompts over your **annotation spreadsheets**, predictions written back **next to
your labels** with the timeline preserved.

Two runs are configured: **(1) `fed_statements_presconf`** — FOMC statements + press-conference
sentences, merged and sorted chronologically by date; **(2) `fed_speeches`** — the speeches file
separately (its dates are year-only, so it is never merged). Each run produces one Excel
workbook in `results/`: your original columns, untouched and in order, plus one prediction
column per model×prompt directly after your `label` column, and a `summary` sheet with the
scores. The models **never see any of your annotations** — only the sentence text is sent
(see the guarantee note below). Pipeline (prompts, sampling settings, retries, resume,
cost tracking) is identical to the verified JSONL version.

## 0. Data files — set the explicit paths
Three explicit paths in the Configuration cell, exactly as before:
`STATEMENTS_PATH` (.xls), `PRESCONF_PATH` (.xlsx), `SPEECHES_PATH` (.xlsx).
Column names are auto-detected (`sentence`/`text`, `label`, `date`/`year`) and every original
column — including the stray `Unnamed:` ones — is preserved in the output. Empty rows are kept
in the output with blank predictions. Outputs land in `results/` as
`fed_statements_presconf_WITH_LLM_LABELS.xlsx` and `fed_speeches_WITH_LLM_LABELS.xlsx`.

In [128]:
!pip install -U anthropic openai pandas openpyxl xlrd

## Packages
```
pip install -U anthropic openai pandas openpyxl xlrd
```
`pandas`+`openpyxl` read/write the .xlsx files; **`xlrd` is required for the legacy `.xls`
statements file** (or re-save it as .xlsx once and update the path). Versions are logged to
`run_manifest.json`.

In [129]:
import json
import os
import random
import re
import sys
import threading
import time
import zlib
import hashlib
import platform
from datetime import datetime, timezone
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

# Global seed: governs few-shot ordering, dry-run fakes, and is sent to every
# API that accepts a seed parameter (see Reproducibility settings).
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
import pandas as pd

## Label assignment — two layers
`LABEL_MAPPING` (digits) drives the prompts and is **unchanged** from the verified pipeline —
identical run-through. `GOLD_TO_DIGIT` translates *your* letter annotations for scoring only:
D→0, H→1, N→2, and the statements file's 5-point labels collapse **MD→D, MH→H** (standard
3-class evaluation). Your original letters are never modified; the merged output adds a
`gold_collapsed` column so the collapse is fully transparent. Unknown labels (e.g. the single
`M` in statements and in presconf) are still classified but excluded from accuracy/F1.

In [130]:
LABEL_MAPPING = {
    0: "Dovish --> Leans towards lower interest rates, more accommodative monetary policy.",
    1: "Hawkish --> Leans towards higher interest rates, tighter monetary policy.",
    2: "Neutral --> Neither strongly dovish nor strongly hawkish, balanced stance.",
}

GOLD_TO_DIGIT = {   # your annotation letters -> scoring digit (MD/MH collapse for statements)
    "D": 0, "MD": 0,
    "H": 1, "MH": 1,
    "N": 2,
}
DIGIT_TO_LETTER = {0: "D", 1: "H", 2: "N"}   # how model predictions are written to Excel

def normalize_gold(v):
    """Map an annotation cell to a scoring digit, or None if unknown/blank."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    return GOLD_TO_DIGIT.get(str(v).strip().upper())

## LLM Models — Anthropic, OpenAI, Mistral, DeepSeek
Unchanged. Pin dated snapshots for the paper run; prices are $/1M tokens for realized-cost tracking.

In [131]:
MODELS = {
    "claude-sonnet": {
        "provider": "anthropic", "model_id": "claude-sonnet-4-6",
        "price_in": 3.00, "price_out": 15.00, "max_out": 4000,
    },
    "claude-haiku": {
        "provider": "anthropic", "model_id": "claude-haiku-4-5",
        "price_in": 1.00, "price_out": 5.00, "max_out": 2000,
    },
    "gpt-5.1": {
        "provider": "openai", "model_id": "gpt-5.1",
        "price_in": 1.25, "price_out": 10.00,
        # 'none' switches off reasoning for GPT-5.1; if the API rejects it,
        # fall back to "minimal". Critical for cost (see study budget).
        "extra": {"reasoning_effort": "none"},
        "no_temperature": True,          # gpt-5 family rejects temperature != 1
        "use_max_completion_tokens": True,
        "max_out": 16,
    },
    "gpt-5-mini": {
        "provider": "openai", "model_id": "gpt-5-mini",
        "price_in": 0.25, "price_out": 2.00,
        "extra": {"reasoning_effort": "minimal"},
        "no_temperature": True,
        "use_max_completion_tokens": True,
        "max_out": 2000,          # ← was 16; headroom for minimal reasoning + the answer
    },
    "mistral-large": {
        "provider": "mistral", "model_id": "mistral-large-latest",
        "price_in": 2.00, "price_out": 6.00,
    },
    "mistral-small": {
        "provider": "mistral", "model_id": "mistral-small-latest",
        "price_in": 0.10, "price_out": 0.30,
    },
    "deepseek-chat": {
        "provider": "deepseek", "model_id": "deepseek-chat",
        "price_in": 0.28, "price_out": 0.42,
    },
    "deepseek-reasoner": {
        "provider": "deepseek", "model_id": "deepseek-reasoner",
        "price_in": 0.28, "price_out": 0.42,
        # max_tokens caps the FINAL answer only; CoT is separate but billed
        # (it shows up in usage.completion_tokens, so costs stay accurate).
        "no_temperature": True,
        "max_out": 4000,
    },
}

## Provider API keys — secure setup
Unchanged: names only in `PROVIDER_ENV`, keys via environment variables / `~/llm_keys.env`
loader / hidden prompt. Never in the notebook.

In [132]:
PROVIDER_ENV = {                 # provider -> name of the environment variable
    "anthropic": "ANTHROPIC_API_KEY",
    "openai":    "OPENAI_API_KEY",
    "mistral":   "MISTRAL_API_KEY",
    "deepseek":  "DEEPSEEK_API_KEY",
}
PROVIDER_BASE_URL = {            # OpenAI-compatible endpoints
    "openai": None,              # SDK default
    "mistral": "https://api.mistral.ai/v1",
    "deepseek": "https://api.deepseek.com",
}

import getpass

def _ensure_keys(providers):
    """Prompt (hidden input) for any missing keys among the given providers."""
    for prov in providers:
        env = PROVIDER_ENV[prov]
        if not os.environ.get(env):
            os.environ[env] = getpass.getpass(f"Enter {env} (input hidden, kept in memory only): ")

In [133]:
# key-file loader (reads ~/llm_keys.env if present; harmless if absent)
KEYFILE = Path.home() / "llm_keys.env"
if KEYFILE.exists():
    for line in KEYFILE.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
    print("key file loaded:", [k for k in PROVIDER_ENV.values() if os.environ.get(k)])

## System prompts
**Byte-identical to the verified pipeline** (frozen; hash recorded in the manifest). The
few-shot prompt draws its 3 labeled examples **cross-source** (see loading cell): the
statements+presconf run uses examples from the *speeches* file, the speeches run uses examples
from the *statements* file — never from the file being evaluated (sentence overlap across your
three files was verified to be zero), so your annotations of the evaluated sentences are never
shown to any model. Set `"fewshot_from": None` in `RUNS` to disable examples entirely.

In [134]:
PROMPTS = {
    # 1. Direct & Concise (Anchored Analyst)
    "p1_analyst_direct": {
        "system": (
            "You are an expert monetary policy analyst. Your job is to classify central bank "
            "communications into monetary policy stances using specific domain guidelines. "
            "Output a single digit corresponding to the correct label and nothing else."
        ),
        "user": (
            "Classify the following sentence based on standard central bank communication rules:\n"
            "- Dovish (0): Decreasing inflation, rising unemployment, low growth projections, dollar appreciation, falling energy/house prices, decreasing yields/spreads, or focus on 'accommodative' / 'maximum employment'.\n"
            "- Hawkish (1): Increasing inflation, falling unemployment, economic output above potential, dollar depreciation, rising energy/house prices, increasing yields/TIPS, or focus on 'price stability' / 'sustained growth'.\n"
            "- Neutral (2): Unchanged/sustained unemployment or growth, reference to foreign nation policies, or neutral phrasing like 'mixed', 'moderate', or 'reaffirmed'.\n\n"
            "Sentence: \"{sentence}\"\n\n"
            "Label digit (0, 1, or 2):"
        ),
    },

    # 2. Comprehensive Guidelines (Detailed Rubric)
    "p2_analyst_rubric": {
        "system": (
            "You are a Senior Monetary Policy Analyst at a central bank specializing in FOMC statement analysis. "
            "You strictly adhere to formal annotation rubrics to label central bank sentiment. "
            "Respond with ONLY a single digit: 0, 1, or 2."
        ),
        "user": (
            "Analyze the policy statement below using the following Annotation Criteria:\n\n"
            "1. DOVISH (0):\n"
            "   - Economic Status: Decreasing inflation, increasing unemployment, low projected growth.\n"
            "   - Prices & Dollar: Dollar appreciates, falling oil/energy/house prices.\n"
            "   - Markets & Fed: Expecting disinflation/subpar inflation, narrowing treasury spreads, decreasing treasury yields, reduction of bank reserves.\n"
            "   - Money/Labor: Low money supply, M2 increase, higher loan demand, increasing productivity.\n"
            "   - Phrasing: Focus on 'accommodative' policy or 'maximum employment'.\n\n"
            "2. HAWKISH (1):\n"
            "   - Economic Status: Increasing inflation, decreasing unemployment, high projected growth, output above potential, falling economic slack.\n"
            "   - Prices & Dollar: Dollar depreciates, rising oil/energy/house prices.\n"
            "   - Markets & Fed: Expecting high inflation, widening treasury spreads, increasing treasury yields/TIPS value, increasing bank reserves.\n"
            "   - Money/Labor: High money supply, increased demand for goods, low demand for loans, decreasing productivity.\n"
            "   - Phrasing: Focus on 'price stability' or 'sustained growth'.\n\n"
            "3. NEUTRAL (2):\n"
            "   - Status/Foreign: Unemployment or growth rate is unchanged/maintained; references to foreign nation trade/economic policies.\n"
            "   - Phrasing: Uses balanced words like 'mixed', 'moderate', or 'reaffirmed'.\n\n"
            "Sentence: \"{sentence}\"\n\n"
            "Digit:"
        ),
    },

    # 3. Rule-Based Chain of Constraint
    "p3_analyst_rules": {
        "system": (
            "You are a central bank research assistant. You map central bank statements to monetary policy stances. "
            "Output only the digit matching the correct stance (0, 1, or 2)."
        ),
        "user": (
            "Task: Classify the stance of the sentence below using central bank annotation rules.\n\n"
            "Annotation Rules:\n"
            "- Classify as 0 (Dovish) if text indicates falling inflation, rising unemployment, economic slack, decreasing yields/energy prices, or mentions 'accommodative'.\n"
            "- Classify as 1 (Hawkish) if text indicates rising inflation, falling unemployment, output above potential, rising yields/energy prices, or mentions 'price stability'.\n"
            "- Classify as 2 (Neutral) if economic variables are maintained/unchanged, refers to foreign trade policies, or uses balanced terms ('mixed', 'moderate', 'reaffirmed').\n"
            "- Stance Priority: Judge only explicit stance expressed in the statement. Prefer 2 (Neutral) if data is reported without clear directional bias.\n\n"
            "Sentence: \"{sentence}\"\n"
            "Label (0, 1, or 2):"
        ),
    },

    # 4. JSON API (Central Bank NLP Pipeline)
    "p4_analyst_json": {
        "system": (
            "You are an automated central bank sentiment classification engine. "
            "Evaluate text against monetary policy annotation criteria and return raw JSON only."
        ),
        "user": (
            "Classify this policy text according to FOMC stance criteria:\n"
            "- 0 (Dovish): Lower rates/inflation, higher unemployment, dollar appreciation, falling yields/energy prices, focus on accommodation.\n"
            "- 1 (Hawkish): Higher rates/inflation, lower unemployment, dollar depreciation, rising yields/energy prices, focus on price stability.\n"
            "- 2 (Neutral): Sustained/unchanged indicators, foreign policy context, or neutral terms ('moderate', 'mixed', 'reaffirmed').\n\n"
            "Sentence: \"{sentence}\"\n\n"
            "Respond ONLY with this format: {\"label\": <digit>}"
        ),
    },

    # 5. Few-Shot (Domain Expert Guided)
    "p5_analyst_fewshot": {
        "system": (
            "You are a senior monetary policy analyst carefully annotating central bank communications. "
            "Reply with a single digit (0, 1, or 2) only."
        ),
        "user": (
            "Use the standard annotation guidelines (Dovish=0: disinflation/slack/accommodation, "
            "Hawkish=1: inflation/tight labor/price stability, Neutral=2: unchanged metrics/foreign policy/mixed terms).\n\n"
            "Examples:\n{examples}\n\n"
            "Now classify this statement:\n"
            "Sentence: \"{sentence}\"\n"
            "Label:"
        ),
    },
}

In [135]:
def _labels_block():
    """Render label definitions from LABEL_MAPPING (single source of truth)."""
    return "\n".join(f"{k} = {v}" for k, v in LABEL_MAPPING.items())

def build_fewshot_examples(df, text_col, label_col, exclude_texts):
    """First D/H/N example from a cross-source dataframe, as digit-labeled demos."""
    picked = {}
    if df is None:
        return ""
    for _, r in df.iterrows():
        g = normalize_gold(r.get(label_col))
        t = r.get(text_col)
        if g is None or not isinstance(t, str) or not t.strip() or t in exclude_texts:
            continue
        # only pure D/H/N rows as demos (skip collapsed MD/MH to keep examples unambiguous)
        if str(r.get(label_col)).strip().upper() in ("D", "H", "N") and g not in picked:
            picked[g] = t.strip()
        if len(picked) == len(LABEL_MAPPING):
            break
    return "\n".join(f'Sentence: "{t}"\nLabel: {k}\n' for k, t in sorted(picked.items()))

def render_prompt(prompt_id, sentence, examples):
    p = PROMPTS[prompt_id]
    user = (p["user"]
            .replace("{labels}", _labels_block())
            .replace("{examples}", examples)
            .replace("{sentence}", sentence.replace('"', "'")))
    return p["system"], user

## Reproducibility settings

Everything that can be pinned, is pinned — set in code in the next cells and summarized here:

| Setting | Value | Applies to |
|---|---|---|
| `temperature` | **0** | Claude Sonnet/Haiku, Mistral Large/Small, DeepSeek-chat |
| `temperature` | *not sent* (API fixed default) | GPT-5.1, GPT-5-mini — the GPT-5 family rejects non-default temperature; determinism is approximated via `seed` + `reasoning_effort` |
| `temperature` | *ignored by API* | DeepSeek-reasoner (documented behaviour) |
| `seed` | **42** (`RANDOM_SEED`) | OpenAI models (`seed=42`, best-effort determinism), Mistral (`random_seed=42`) — Anthropic and DeepSeek expose no seed parameter |
| `top_p` / other samplers | provider defaults, untouched | all — mixing `top_p` with `temperature=0` is discouraged and adds nothing |
| `reasoning_effort` | `"none"` (GPT-5.1) / `"minimal"` (GPT-5-mini) | pins reasoning off — also the difference between ~$0.72 and ~$7.60 on the test run |
| max output tokens | 8 (16 for GPT-5.x, 32 for DeepSeek-reasoner's final answer) | all — forces label-only outputs |
| Python RNG | `random.seed(42)`; dry-run fakes use `zlib.crc32` (stable across sessions, unlike `hash()`) | pipeline itself |
| Few-shot examples | first sentence of each class in `FEWSHOT_FILE`, in label order | p5 prompt |
| Model versions | pin dated snapshots for the paper run | all |

**Honest caveat for the methods section:** `temperature=0` plus a seed makes outputs *highly stable*
but does **not** guarantee bit-identical responses — providers batch requests on changing hardware,
and hosted models can be silently updated. That is why true reproducibility here rests on:
(1) pinned dated model snapshots, (2) `run_manifest.json` (seed, model IDs, prompt hash, data
checksums, package versions, timestamp), and (3) the fact that **every raw response is stored** in
`results/*.jsonl`, so all metrics can be recomputed from disk without re-calling any API.
Report macro-F1 as mean ± SD across the five prompts — the summary cell prints exactly that.

In [136]:
class RateLimiter:
    """Simple global throttle: at most `rpm` requests/minute per provider."""
    def __init__(self, rpm):
        self.interval = 60.0 / max(rpm, 1)
        self.lock = threading.Lock()
        self.next_ok = 0.0
    def wait(self):
        with self.lock:
            now = time.time()
            wait_for = max(0.0, self.next_ok - now)
            self.next_ok = max(now, self.next_ok) + self.interval
        if wait_for > 0:
            time.sleep(wait_for)

mistral_slot = threading.Semaphore(4)   # cap Mistral to 1 request in flight (free-tier concurrency limit)

_clients = {}
def get_client(provider):
    if provider in _clients:
        return _clients[provider]
    key = os.environ.get(PROVIDER_ENV[provider])
    if not key:
        raise RuntimeError(f"Missing {PROVIDER_ENV[provider]} -- set the environment "
                           f"variable or run _ensure_keys([...]) in the API-keys cell.")
    if provider == "anthropic":
        import anthropic
        _clients[provider] = anthropic.Anthropic(api_key=key)
    else:
        from openai import OpenAI
        base = PROVIDER_BASE_URL[provider]
        _clients[provider] = OpenAI(api_key=key, base_url=base) if base else OpenAI(api_key=key)
    return _clients[provider]

def call_model(mkey, system, user, dry_run=False):
    """Returns (raw_text, in_tokens, out_tokens). Raises on final failure."""
    cfg = MODELS[mkey]
    if dry_run:  # deterministic fake: lets you test the pipeline with no keys
        fake = str(random.Random(zlib.crc32(user.encode())).choice(list(LABEL_MAPPING)))
        return fake, int(len(system + user) / 4) + 7, 5

    provider, max_out = cfg["provider"], cfg.get("max_out", 8)
    last_err = None
    for attempt in range(6):
        try:
            if provider == "anthropic":
                r = get_client(provider).messages.create(
                    model=cfg["model_id"], max_tokens=max_out, temperature=0,
                    system=system, messages=[{"role": "user", "content": user}],
                )
                return (r.content[0].text if r.content else ""), r.usage.input_tokens, r.usage.output_tokens
            kw = dict(model=cfg["model_id"],
                      messages=[{"role": "system", "content": system},
                                {"role": "user", "content": user}])
            if cfg.get("use_max_completion_tokens"):
                kw["max_completion_tokens"] = max_out
            else:
                kw["max_tokens"] = max_out
            if not cfg.get("no_temperature"):
                kw["temperature"] = 0
            if provider == "openai":
                kw["seed"] = RANDOM_SEED          # best-effort determinism (OpenAI)
            elif provider == "mistral":
                kw["extra_body"] = {"random_seed": RANDOM_SEED}   # Mistral's seed parameter
            kw.update(cfg.get("extra", {}))
            if provider == "mistral":
                with mistral_slot:                # only 1 Mistral call in flight at once
                    r = get_client(provider).chat.completions.create(**kw)
            else:
                r = get_client(provider).chat.completions.create(**kw)
            txt = r.choices[0].message.content or ""
            # note: for deepseek-reasoner, usage.completion_tokens includes CoT
            return txt, r.usage.prompt_tokens, r.usage.completion_tokens
        except Exception as e:  # 429s, 5xx, transient network -- back off & retry
            last_err = e
            time.sleep(min(2 ** attempt * 2, 60) + random.random())
    raise RuntimeError(f"{mkey}: failed after retries: {last_err}")

## Parsing & metrics
Unchanged (robust digit extraction + manual macro-F1); metrics only count rows whose gold label maps to D/H/N.

In [137]:
_valid = "".join(str(k) for k in LABEL_MAPPING)

def parse_label(raw):
    if not raw:
        return None
    m = re.search(r'"label"\s*:\s*([%s])' % _valid, raw)       # JSON form
    if m:
        return int(m.group(1))
    m = re.search(r"\b([%s])\b" % _valid, raw)                 # bare digit
    if m:
        return int(m.group(1))
    low = raw.lower()                                          # word fallback
    for k, v in LABEL_MAPPING.items():
        if v.split()[0].lower() in low:
            return k
    return None

def macro_f1(golds, preds):
    scores = []
    for c in LABEL_MAPPING:
        tp = sum(1 for g, p in zip(golds, preds) if g == c and p == c)
        fp = sum(1 for g, p in zip(golds, preds) if g != c and p == c)
        fn = sum(1 for g, p in zip(golds, preds) if g == c and p != c)
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        scores.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    return sum(scores) / len(scores)

## Excel loading & the annotation-privacy guarantee
`load_source` reads each spreadsheet (any `Unnamed:` legend columns and empty rows preserved for
output). `build_items` extracts, per row, **only** `text` (the sentence), the scoring digit, and
bookkeeping (`source`, `orig_row`, sort date). The prompt renderer receives `item["text"]` and
nothing else — your `label` column physically cannot reach a request. Merged run: statements +
presconf concatenated, stable-sorted by date (ties keep statements-then-presconf, original order
within each file).

In [138]:
def load_source(path):
    path = Path(path)
    try:
        df = pd.read_excel(path)          # engine auto: openpyxl for .xlsx, xlrd for .xls
    except ImportError as e:
        raise ImportError(f"{path.name}: reading legacy .xls needs the 'xlrd' package -- "
                          f"run: %pip install -U xlrd   (or re-save the file as .xlsx)") from e
    return df

def _detect(df, cands):
    for c in df.columns:
        if str(c).strip().lower() in cands:
            return c
    return None

def build_items(run_cfg):
    """Returns (items, frames). items: dicts with text/gold/source/orig_row.
    frames: {source_tag: (df, text_col, label_col, date_col)} for output writing."""
    frames, rows = {}, []
    for tag, path in run_cfg["sources"]:
        df = load_source(path)
        tcol = _detect(df, {"sentence", "text"})
        lcol = _detect(df, {"label"})
        dcol = _detect(df, {"date", "year"})
        assert tcol, f"{path}: no sentence/text column found"
        frames[tag] = (df, tcol, lcol, dcol)
        d = pd.to_datetime(df[dcol], errors="coerce") if dcol is not None else pd.Series([pd.NaT]*len(df))
        for i in range(len(df)):
            t = df.at[i, tcol]
            if not isinstance(t, str) or not t.strip():
                continue                                   # empty row: kept in output, no call
            rows.append({"source": tag, "orig_row": i, "text": t.strip(),
                         "label": normalize_gold(df.at[i, lcol]) if lcol is not None else None,
                         "_d": d.iloc[i]})
    if run_cfg.get("sort_by_date"):
        rows.sort(key=lambda r: (pd.Timestamp.max if pd.isna(r["_d"]) else r["_d"],))
    for r in rows:
        r.pop("_d")
    return rows, frames

def items_fingerprint(items):
    h = hashlib.sha256()
    for r in items:
        h.update(f"{r['source']}|{r['orig_row']}|{r['text']}".encode())
    return h.hexdigest()

## Runner — resume + incremental writes
Same engine; result files are now per **run** (`results__{run}__{model}__{prompt}.jsonl`) and each row carries `source`/`orig_row` so predictions can be joined back onto your exact spreadsheet rows at any time, even mid-run.

In [139]:
def run_one(run_tag, mkey, prompt_id, items, examples, outdir, limiter, workers, dry_run):
    outfile = outdir / f"results__{run_tag}__{mkey}__{prompt_id}.jsonl"
    done = set()
    if outfile.exists():
        with open(outfile) as fh:
            for line in fh:
                try:
                    done.add(json.loads(line)["idx"])
                except Exception:
                    pass
    todo = [(i, it) for i, it in enumerate(items) if i not in done]
    if not todo:
        print(f"  {run_tag}: {mkey} x {prompt_id}: already complete ({len(done)} rows)")
        return
    print(f"  {run_tag}: {mkey} x {prompt_id}: {len(todo)} to do ({len(done)} resumed)")
    lock = threading.Lock()
    cfg = MODELS[mkey]
 
    def work(idx, item):
        if not dry_run:
            limiter.wait()
        system, user = render_prompt(prompt_id, item["text"], examples)
        try:
            raw, tin, tout = call_model(mkey, system, user, dry_run)
            err = None
        except Exception as e:
            raw, tin, tout, err = "", 0, 0, str(e)
        row = {
            "idx": idx, "model": mkey, "prompt": prompt_id,
            "source": item["source"], "orig_row": item["orig_row"],
            "gold": item["label"], "pred": parse_label(raw), "raw": raw.strip(),
            "in_tok": tin, "out_tok": tout,
            "cost": (tin * cfg["price_in"] + tout * cfg["price_out"]) / 1e6,
            "error": err,
        }
        with lock:
            with open(outfile, "a") as fh:
                fh.write(json.dumps(row) + "\n")
        return err
 
    n_err = 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = [ex.submit(work, i, it) for i, it in todo]
        for k, f in enumerate(as_completed(futures), 1):
            if f.result():
                n_err += 1
            if k % 50 == 0 or k == len(futures):
                print(f"    {mkey} x {prompt_id}: {k}/{len(futures)} ({n_err} errors)", flush=True)
 
def summarize(outdir, run_tag):
    rows = []
    for f in sorted(outdir.glob(f"results__{run_tag}__*.jsonl")):
        recs = [json.loads(l) for l in open(f) if l.strip()]
        if not recs:
            continue
        ok = [r for r in recs if r["pred"] is not None and not r["error"] and r["gold"] is not None]
        golds = [r["gold"] for r in ok]
        preds = [r["pred"] for r in ok]
        acc = sum(g == p for g, p in zip(golds, preds)) / len(ok) if ok else 0.0
        rows.append({
            "model": recs[0]["model"], "prompt": recs[0]["prompt"], "n": len(recs),
            "parsed": len(ok), "errors": sum(1 for r in recs if r["error"]),
            "accuracy": round(acc, 4),
            "macro_f1": round(macro_f1(golds, preds), 4) if ok else 0.0,
            "in_tok": sum(r["in_tok"] for r in recs),
            "out_tok": sum(r["out_tok"] for r in recs),
            "cost_usd": round(sum(r["cost"] for r in recs), 4),
        })
    if not rows:
        print("No results yet.")
        return
    import csv
    with open(outdir / f"summary__{run_tag}.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=rows[0].keys())
        w.writeheader(); w.writerows(rows)
    hdr = f"{'model':18s} {'prompt':12s} {'n':>5s} {'acc':>7s} {'macroF1':>8s} {'out_tok':>9s} {'cost$':>9s}"
    print("\n" + hdr + "\n" + "-" * len(hdr))
    for r in rows:
        print(f"{r['model']:18s} {r['prompt']:12s} {r['n']:5d} {r['accuracy']:7.3f} "
              f"{r['macro_f1']:8.3f} {r['out_tok']:9d} {r['cost_usd']:9.4f}")
    # per-model mean +/- sd across prompts (prompt-sensitivity, nice for the paper)
    by_model = defaultdict(list)
    for r in rows:
        by_model[r["model"]].append(r["macro_f1"])
    print("\nMacro-F1 across prompts (mean +/- sd):")
    for m, v in by_model.items():
        mean = sum(v) / len(v)
        sd = (sum((x - mean) ** 2 for x in v) / len(v)) ** 0.5
        print(f"  {m:18s} {mean:.3f} +/- {sd:.3f}  (k={len(v)})")
    total = sum(r["cost_usd"] for r in rows)
    print(f"\nTotal realized cost so far: ${total:.2f}")
    print(f"Summary written to {outdir}/summary__{run_tag}.csv")

## Excel output — your file + prediction columns
Rebuilds each run's workbook from the incremental logs (safe to re-run any time, even mid-run):
sheet **data** = your original columns in original/merged-timeline order, then right after your
`label` column: `gold_collapsed`, `llm_majority`, and one letter column (D/H/N) per model×prompt;
sheet **summary** = accuracy & macro-F1 per combination plus run metadata. Cells are sanitized
for Excel's character rules; nothing in your original data is altered.

In [140]:
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

def _clean(v):
    return ILLEGAL_CHARACTERS_RE.sub("", v) if isinstance(v, str) else v

def write_excel(run_tag, run_cfg, outdir):
    items, frames = build_items(run_cfg)

    # collect predictions from the incremental logs: (source, orig_row) -> {combo: letter}
    preds, combos = defaultdict(dict), []
    for rf in sorted(outdir.glob(f"results__{run_tag}__*.jsonl")):
        for line in open(rf):
            r = json.loads(line)
            combo = f'{r["model"]}__{r["prompt"]}'
            if combo not in combos:
                combos.append(combo)
            letter = DIGIT_TO_LETTER.get(r["pred"], "?" if r["pred"] is None and not r["error"] else "ERR")
            preds[(r["source"], r["orig_row"])][combo] = letter
    combos.sort()

    # assemble per-source frames with prediction columns
    outs = []
    for tag, (df, tcol, lcol, dcol) in frames.items():
        d = df.copy()
        if lcol is not None:
            # normalize the 5-point scale to 3 classes in the OUTPUT (source file untouched)
            d[lcol] = d[lcol].replace({"MD": "D", "MH": "H"})
        d.insert(0, "source", tag)
        d.insert(1, "orig_row", range(len(d)))
        gc = [{0: "D", 1: "H", 2: "N"}.get(normalize_gold(v), "")
              for v in (d[lcol] if lcol is not None else [None] * len(d))]
        anchor = list(d.columns).index(lcol) + 1 if lcol is not None else len(d.columns)
        d.insert(anchor, "gold_collapsed", gc)
        rows_pred = [preds.get((tag, i), {}) for i in range(len(d))]
        maj = []
        for p in rows_pred:
            vals = [v for v in p.values() if v in ("D", "H", "N")]
            maj.append(max(set(vals), key=vals.count) if vals else "")
        d.insert(anchor + 1, "llm_majority", maj)
        for j, combo in enumerate(combos):
            d.insert(anchor + 2 + j, combo, [p.get(combo, "") for p in rows_pred])
        d["_sortdate"] = pd.to_datetime(d[dcol], errors="coerce") if dcol is not None else pd.NaT
        outs.append(d)

    full = pd.concat(outs, ignore_index=True, sort=False)
    if run_cfg.get("sort_by_date"):
        full = full.sort_values("_sortdate", kind="stable", na_position="last").reset_index(drop=True)
    full = full.drop(columns=["_sortdate"])
    full = full.map(_clean)

    # summary sheet from the logs
    srows = []
    for rf in sorted(outdir.glob(f"results__{run_tag}__*.jsonl")):
        recs = [json.loads(l) for l in open(rf) if l.strip()]
        if not recs:
            continue
        ok = [r for r in recs if r["pred"] is not None and not r["error"] and r["gold"] is not None]
        g = [r["gold"] for r in ok]
        p = [r["pred"] for r in ok]
        srows.append({"model": recs[0]["model"], "prompt": recs[0]["prompt"],
                      "n_calls": len(recs), "n_scored": len(ok),
                      "accuracy": round(sum(a == b for a, b in zip(g, p)) / len(ok), 4) if ok else "",
                      "macro_f1": round(macro_f1(g, p), 4) if ok else "",
                      "in_tok": sum(r["in_tok"] for r in recs),
                      "out_tok": sum(r["out_tok"] for r in recs),
                      "cost_usd": round(sum(r["cost"] for r in recs), 4)})
    sdf = pd.DataFrame(srows)

    out = outdir / f"{run_tag}_WITH_LLM_LABELS.xlsx"
    with pd.ExcelWriter(out, engine="openpyxl") as xw:
        full.to_excel(xw, sheet_name="data", index=False)
        (sdf if len(sdf) else pd.DataFrame([{"note": "no results yet"}])).to_excel(
            xw, sheet_name="summary", index=False)
        for ws in xw.book.worksheets:
            for cell in ws[1]:
                cell.font = Font(name="Arial", bold=True)
            ws.freeze_panes = "A2"
        ws = xw.book["data"]
        for j, col in enumerate(full.columns, start=1):
            w = 12
            if str(col) in ("sentence", "text"):
                w = 80
            elif str(col) in ("raw_text", "clean_text"):
                w = 40
            ws.column_dimensions[get_column_letter(j)].width = w
    print(f"{run_tag}: wrote {out}  ({len(full)} rows x {len(full.columns)} cols; "
          f"{len(combos)} prediction columns)")
    return out

In [141]:
import json
manifest = json.loads((OUTDIR / "run_manifest.json").read_text())
old_fp = manifest["runs"]["fed_statements_presconf"]["items_sha256"]

items, _ = build_items(RUNS["fed_statements_presconf"])
if LIMIT:
    items = items[:LIMIT]
new_fp = items_fingerprint(items)

print("sentence/order fingerprint match:", old_fp == new_fp)

sentence/order fingerprint match: True


## Configuration — edit here, nothing else
Explicit paths to your three annotation files, two pre-configured runs (statements+presconf
merged by date; speeches separate). Pilot with `LIMIT = 25`, then `LIMIT = None`.

In [142]:
# --- EXPLICIT DATA FILE PATHS: edit these three lines to your machine ---------
STATEMENTS_PATH = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/data/FED/Annotated/FOMC STATEMENTS/fomc_statements_train_1000.xls")
PRESCONF_PATH   = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/data/FED/Annotated/Press Conferences/presconf_to_annotate.xlsx")
SPEECHES_PATH   = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/data/FED/Annotated/Speeches/Annotated with Shah Merge/lab-manual-sp-train-5768-augmented.xlsx")

RUNS = {
    "fed_statements_presconf": {          # merged chronologically (both have day-level dates)
        "sources": [("statements", STATEMENTS_PATH), ("presconf", PRESCONF_PATH)],
        "sort_by_date": True,
        "fewshot_from": SPEECHES_PATH,    # cross-source demos; None = no examples in p5
    },
    "fed_speeches": {                     # year-only dates -> annotated separately, original order
        "sources": [("speeches", SPEECHES_PATH)],
        "sort_by_date": False,
        "fewshot_from": STATEMENTS_PATH,
    },
}

RUN_MODELS  = list(MODELS)
RUN_PROMPTS = list(PROMPTS)
LIMIT       = None         # pilot on first N sentences per run; None = full run
WORKERS     = 4
RPM         = 200
OUTDIR      = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/Extension LLM Classification Task/Fed Side/Results")
DRY_RUN     = False
ALLOW_INPUT_CHANGE = True   # one-run override for harmless input edits (see Preflight)

## Preflight — file check, alignment verification, run manifest
Validates paths and logs versions/checksums as before, but the resume guard is now
**alignment-based**: it re-checks that every already-written result row still maps to the same
spreadsheet row (`source`, `orig_row`) under the current configuration. Changing `LIMIT` in
either direction therefore passes automatically — pilot → full run just resumes. A hard stop
occurs only for the genuinely dangerous cases: existing results that no longer line up
(rows added/removed/reordered in an input), or an input file whose checksum changed since
results were written. For deliberate, harmless edits (e.g. you corrected a label — sentences
and row order intact), set `ALLOW_INPUT_CHANGE = True` in the Configuration cell for one run.

In [143]:
def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def verify_alignment(tag, items, outdir):
    """Every existing result row must still point at the same spreadsheet row."""
    key_of = {i: (it["source"], it["orig_row"]) for i, it in enumerate(items)}
    checked = mismatched = beyond = 0
    for rf in sorted(outdir.glob(f"results__{tag}__*.jsonl")):
        for line in open(rf):
            try:
                r = json.loads(line)
            except Exception:
                continue
            checked += 1
            k = key_of.get(r["idx"])
            if k is None:
                beyond += 1          # row outside the current LIMIT window: harmless
            elif k != (r["source"], r["orig_row"]):
                mismatched += 1
    if mismatched:
        raise RuntimeError(
            f"{tag}: {mismatched} existing result rows no longer line up with the current item "
            f"order (rows were added/removed/reordered in an input file). Either restore the "
            f"original inputs, or delete results__{tag}__*.jsonl to reclassify this run.")
    return checked, beyond

OUTDIR.mkdir(exist_ok=True)
missing = [p for p in (STATEMENTS_PATH, PRESCONF_PATH, SPEECHES_PATH) if not Path(p).exists()]
if missing:
    raise FileNotFoundError(f"Not found: {[str(m) for m in missing]} -- edit the three *_PATH "
                            f"lines in the Configuration cell (see section 0).")

# previous manifest (if any) for input-change detection
prev_runs = {}
mpath = OUTDIR / "run_manifest.json"
if mpath.exists():
    try:
        prev_runs = json.loads(mpath.read_text()).get("runs", {})
    except Exception:
        prev_runs = {}
_allow_change = bool(globals().get("ALLOW_INPUT_CHANGE", False))

manifest = {"timestamp_utc": datetime.now(timezone.utc).isoformat(), "random_seed": RANDOM_SEED,
            "python": sys.version.split()[0], "platform": platform.platform(), "packages": {},
            "models": {k: MODELS[k]["model_id"] for k in RUN_MODELS},
            "prompts_sha256": hashlib.sha256(json.dumps({k: PROMPTS[k] for k in RUN_PROMPTS},
                                                        sort_keys=True).encode()).hexdigest(),
            "gold_to_digit": GOLD_TO_DIGIT, "runs": {},
            "config": {"limit": LIMIT, "rpm": RPM, "workers": WORKERS, "dry_run": DRY_RUN,
                       "allow_input_change": _allow_change}}
for pkg in ("anthropic", "openai", "pandas", "openpyxl", "xlrd"):
    try:
        manifest["packages"][pkg] = __import__(pkg).__version__
    except Exception:
        manifest["packages"][pkg] = "NOT INSTALLED"

RUN_ITEMS = {}
for tag, cfg in RUNS.items():
    file_hashes = {str(p): _sha256(p) for _, p in cfg["sources"]}
    has_results = any(OUTDIR.glob(f"results__{tag}__*.jsonl"))
    prev_hashes = prev_runs.get(tag, {}).get("files", {})
    changed = [p for p, h in file_hashes.items()
               if (p in prev_hashes and prev_hashes[p] != h)
               or (prev_hashes and p not in prev_hashes and h not in prev_hashes.values())]
    if changed and has_results and not _allow_change:
        raise RuntimeError(
            f"{tag}: input file(s) changed since existing results were written: {changed}. "
            f"If the edit was harmless for alignment (labels/other columns only; sentences and "
            f"row order intact), set ALLOW_INPUT_CHANGE = True in the Configuration cell and "
            f"rerun. Otherwise restore the file or delete results__{tag}__*.jsonl.")
    items, _ = build_items(cfg)
    if LIMIT:
        items = items[:LIMIT]
    checked, beyond = verify_alignment(tag, items, OUTDIR)
    RUN_ITEMS[tag] = items
    dist = defaultdict(int)
    for it in items:
        dist[{0: "D", 1: "H", 2: "N"}.get(it["label"], "unscored")] += 1
    manifest["runs"][tag] = {"n_items": len(items), "items_sha256": items_fingerprint(items),
                             "files": file_hashes, "fewshot_from": str(cfg.get("fewshot_from")),
                             "gold_dist": dict(dist)}
    note = f"; verified {checked} existing result rows aligned" if checked else ""
    note += f" ({beyond} outside current LIMIT window)" if beyond else ""
    print(f"{tag}: {len(items)} sentences to classify, gold distribution {dict(dist)}{note}")

for f in OUTDIR.glob("*__items.sha256"):   # retire files from the old fingerprint guard
    f.unlink()
with open(OUTDIR / "run_manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2)
print(f"packages: {manifest['packages']}")
print(f"manifest written -> {OUTDIR/'run_manifest.json'}")

fed_statements_presconf: 1998 sentences to classify, gold distribution {'H': 491, 'N': 877, 'D': 628, 'unscored': 2}; verified 84999 existing result rows aligned
fed_speeches: 951 sentences to classify, gold distribution {'D': 201, 'N': 520, 'H': 230}; verified 35909 existing result rows aligned
packages: {'anthropic': '0.120.2', 'openai': '2.53.0', 'pandas': '3.0.5', 'openpyxl': '3.1.5', 'xlrd': '2.0.2'}
manifest written -> /Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/Extension LLM Classification Task/Fed Side/Results/run_manifest.json


In [144]:
import os
print(os.getcwd())

/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project


## Run
Both runs execute back-to-back; resume works per run. Each run finishes by writing its Excel —
and you can re-execute just the last loop's `write_excel` line at any moment to get a snapshot
workbook of whatever is done so far.

In [145]:
providers_needed = {MODELS[m]["provider"] for m in RUN_MODELS}
if not DRY_RUN:
    _ensure_keys(sorted(providers_needed))
    for p in providers_needed:
        get_client(p)

limiters = {p: RateLimiter(RPM) for p in PROVIDER_ENV}
limiters["mistral"] = RateLimiter(RPM)   # real cap: 0.83 req/s (49.8/min) AND 25,000 tok/min
total = sum(len(v) for v in RUN_ITEMS.values()) * len(RUN_MODELS) * len(RUN_PROMPTS)
print(f"Plan: {len(RUN_MODELS)} models x {len(RUN_PROMPTS)} prompts x "
      f"{ {t: len(v) for t, v in RUN_ITEMS.items()} } sentences = {total:,} calls"
      f"{'  [DRY RUN]' if DRY_RUN else ''}\n")

for tag, cfg in RUNS.items():
    items = RUN_ITEMS[tag]
    ex_df = load_source(cfg["fewshot_from"]) if cfg.get("fewshot_from") else None
    examples = build_fewshot_examples(ex_df, _detect(ex_df, {"sentence","text"}) if ex_df is not None else None,
                                      _detect(ex_df, {"label"}) if ex_df is not None else None,
                                      exclude_texts={it["text"] for it in items})
    for mkey in RUN_MODELS:
        for pid in RUN_PROMPTS:
            run_one(tag, mkey, pid, items, examples, OUTDIR,
                    limiters[MODELS[mkey]["provider"]], WORKERS, DRY_RUN)
    summarize(OUTDIR, tag)
    write_excel(tag, cfg, OUTDIR)

Plan: 8 models x 5 prompts x {'fed_statements_presconf': 1998, 'fed_speeches': 951} sentences = 117,960 calls

  fed_statements_presconf: claude-sonnet x p1_analyst_direct: 5 to do (1993 resumed)


KeyboardInterrupt: 

In [ ]:
for rf in OUTDIR.glob("results__*.jsonl"):
    lines = [l for l in open(rf) if l.strip()]
    keep = [l for l in lines if json.loads(l).get("error") is None]
    if len(keep) != len(lines):
        open(rf, "w").writelines(keep)
        print(rf.name, "-> removed", len(lines) - len(keep), "failed rows for retry")

In [ ]:
for rf in OUTDIR.glob("results__*.jsonl"):
    lines = [l for l in open(rf) if l.strip()]
    keep = [l for l in lines if not (json.loads(l).get("pred") is None
                                     and json.loads(l).get("error") is None)]
    if len(keep) != len(lines):
        open(rf, "w").writelines(keep)
        print(rf.name, "-> removed", len(lines) - len(keep), "unparsed rows for retry")

to_reheal = {tuple(rf.stem.split("__")[1:]) for rf in OUTDIR.glob("results__*.jsonl")
             if not all(json.loads(l).get("pred") is not None for l in open(rf) if l.strip())}

if not to_reheal:
    print("No unparsed rows found.")
else:
    examples_cache = {}
    for tag in {t for t, _, _ in to_reheal}:
        cfg = RUNS[tag]
        ex_df = load_source(cfg["fewshot_from"]) if cfg.get("fewshot_from") else None
        examples_cache[tag] = build_fewshot_examples(
            ex_df, _detect(ex_df, {"sentence", "text"}) if ex_df is not None else None,
            _detect(ex_df, {"label"}) if ex_df is not None else None,
            exclude_texts={it["text"] for it in RUN_ITEMS[tag]})
    for tag, mkey, prompt_id in sorted(to_reheal):
        run_one(tag, mkey, prompt_id, RUN_ITEMS[tag], examples_cache[tag], OUTDIR,
                limiters[MODELS[mkey]["provider"]], WORKERS, DRY_RUN)
    for tag in {t for t, _, _ in to_reheal}:
        summarize(OUTDIR, tag)
        write_excel(tag, RUNS[tag], OUTDIR)

No unparsed rows found.


In [146]:
OUTDIR      = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/Extension LLM Classification Task/Fed Side/Results")
Outdir_excel_file = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/Extension LLM Classification Task/Fed Side/Excel Sentences Results")
OUTDIR.mkdir(parents=True, exist_ok=True)
Outdir_excel_file.mkdir(parents=True, exist_ok=True)

import shutil

def dedupe_results(outdir):
    """Remove duplicate idx rows from every results file (keeps the last occurrence).
    Cause: interrupting the kernel without a full restart can leave a background
    worker thread writing a row after the next run already resumed past it."""
    total_removed = 0
    for rf in sorted(outdir.glob("results__*.jsonl")):
        lines = [json.loads(l) for l in open(rf) if l.strip()]
        n_unique = len({r["idx"] for r in lines})
        if len(lines) == n_unique:
            continue
        last_by_idx = {}
        for r in lines:
            last_by_idx[r["idx"]] = r        # later occurrence wins
        clean = [last_by_idx[i] for i in sorted(last_by_idx)]
        open(rf, "w").writelines(json.dumps(r) + "\n" for r in clean)
        removed = len(lines) - len(clean)
        total_removed += removed
        print(f"{rf.name}: removed {removed} duplicate rows ({len(lines)} -> {len(clean)})")
    if total_removed == 0:
        print("Dedupe: no duplicates found.")
    else:
        print(f"\nDedupe: {total_removed} duplicate rows removed total.")

dedupe_results(OUTDIR)

for tag, cfg in RUNS.items():
    xlsx_path = write_excel(tag, cfg, OUTDIR)         # reads jsonl + writes xlsx into OUTDIR
    dest = Outdir_excel_file / xlsx_path.name
    shutil.move(str(xlsx_path), str(dest))            # relocate the finished workbook
    print(f"{tag}: moved workbook -> {dest}")

results__fed_statements_presconf__gpt-5-mini__p1_analyst_direct.jsonl: removed 5382 duplicate rows (7380 -> 1998)
results__fed_statements_presconf__mistral-large__p1_analyst_direct.jsonl: removed 164 duplicate rows (2162 -> 1998)

Dedupe: 5546 duplicate rows removed total.
fed_statements_presconf: wrote /Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/Extension LLM Classification Task/Fed Side/Results/fed_statements_presconf_WITH_LLM_LABELS.xlsx  (1999 rows x 58 cols; 40 prediction columns)
fed_statements_presconf: moved workbook -> /Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/Extension LLM Classification Task/Fed Side/Excel Sentences Results/fed_statements_presconf_WITH_LLM_LABELS.xlsx
fed_speeches: wrote /Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/Extension LLM Classification Task/Fed Side/Results/fed_speeches_WITH_LLM_LABELS.xlsx  (951 rows x 48 cols; 38 prediction columns)
fed_spee

In [147]:
dedupe_results(OUTDIR)

Dedupe: no duplicates found.


In [148]:
summarize(OUTDIR, "fed_statements_presconf")
summarize(OUTDIR, "fed_speeches")


model              prompt           n     acc  macroF1   out_tok     cost$
--------------------------------------------------------------------------
claude-haiku       p1_analyst_direct  1998   0.656    0.651      9990    0.5545
claude-haiku       p2_analyst_rubric  1998   0.675    0.669      9990    0.9241
claude-haiku       p3_analyst_rules  1998   0.643    0.635     15948    0.6182
claude-haiku       p4_analyst_json  1998   0.636    0.616     15984    0.4725
claude-haiku       p5_analyst_fewshot  1998   0.608    0.606      9990    0.6444
claude-sonnet      p1_analyst_direct  1998   0.649    0.644     10403    1.6755
claude-sonnet      p2_analyst_rubric  1998   0.650    0.647      9990    2.7782
claude-sonnet      p3_analyst_rules  1549   0.685    0.675      7778    1.3694
claude-sonnet      p4_analyst_json  1998   0.622    0.621     15984    1.4236
claude-sonnet      p5_analyst_fewshot  1998   0.575    0.574      9990    1.9391
deepseek-chat      p1_analyst_direct  1998   0.628   